# 🔧 核心函数详解：`parse_model(d, ch)` — 从 YAML 到模型的转换器

## 一句话总结

> **`parse_model()` 读取 `.yaml` 配置中的 `backbone` 和 `head` 列表，逐行解析，每一行变成一个实际的 PyTorch 层/module，最后组装成一个 `nn.Sequential` 模型。**

它是 YOLOv5 **"配置驱动"** 设计思想的灵魂 —— 修改模型结构只需要改 `.yaml`，不需要动任何代码。

---

## 一、调用入口

```python
# DetectionModel.__init__() 中：
self.model, self.save = parse_model(deepcopy(self.yaml), ch=[ch])
```

| 参数 | 含义 |
|---|---|
| `d` | 从 `.yaml` 读出的字典（包含 `backbone`、`head`、`nc`、`depth_multiple`、`width_multiple`、`anchors` 等） |
| `ch` | 输入通道列表，初始为 `[3]`（RGB 图片 3 通道），后续逐层更新 |

---

## 二、函数全貌（68 行代码，三大阶段）

```mermaid
graph TD
    A["parse_model(d, ch) 开始"] --> B["阶段1: 提取全局参数"]
    B --> C["提取 anchors, nc, gd, gw, act"]
    C --> D["阶段2: 逐行解析 backbone+head"]
    D --> E["for 循环遍历每一层配置"]
    E --> F["Step A: eval 字符串 -> 实际类"]
    F --> G["Step B: 计算深度 n (深度系数 gd)"]
    G --> H["Step C: 计算通道 c2 (宽度系数 gw)"]
    H --> I["Step D: 实例化模块 m_"]
    I --> J["Step E: 附加元信息 i, f, type, np"]
    J --> K["Step F: 更新通道列表 ch"]
    K --> L["阶段3: 返回"]
    L --> M["返回 nn.Sequential(*layers), sorted(save)"]
```

---

## 三、阶段 1：提取全局参数

```python
anchors, nc, gd, gw, act = d['anchors'], d['nc'], d['depth_multiple'], d['width_multiple'], d.get('activation')
```

从 `yolov5s.yaml` 来看，提取的值是：

| 参数 | 值 | 含义 |
|---|---|---|
| `nc` | 80 | 类别数（COCO 标准） |
| `gd` (depth_multiple) | **0.33** | 深度系数 → 控制 C3 中 Bottleneck 重复次数 |
| `gw` (width_multiple) | **0.50** | 宽度系数 → 控制卷积输出通道数 |
| `anchors` | 3 组 × 3 个（共 9 个） | P3/P4/P5 检测层的预设锚框 |

> 💡 `gd=0.33` + `gw=0.50` 就是 **yolov5s** 的缩放因子。换成 `gd=1.0` + `gw=1.0` 就是 yolov5l，完全不需要改代码。

---

## 四、阶段 2：逐行解析（for 循环体）

这是核心中的核心，逐行处理 `backbone` 和 `head` 列表：

```python
for i, (f, n, m, args) in enumerate(d['backbone'] + d['head']):
```

以 `yolov5s.yaml` 第 0 行为例：

```yaml
# [from, number, module, args]
  [-1, 1, Conv, [64, 6, 2, 2]],   # 0-P1/2
```

- `i = 0`（当前层索引号）
- `f = -1`（from，表示从上一层取输入）
- `n = 1`（number，该模块重复次数）
- `m = 'Conv'`（module 名称字符串）
- `args = [64, 6, 2, 2]`（参数列表）

### Step A: 字符串 → 实际类

```python
m = eval(m) if isinstance(m, str) else m
```

`'Conv'` → `models.common.Conv` 这个类

同时参数也做同样的 eval 转换（比如 `'nn.Upsample'` → `torch.nn.Upsample`）。

### Step B: 深度系数调整

```python
n = n_ = max(round(n * gd), 1) if n > 1 else n
```

| 配置文件 n | gd=0.33 | 实际 n | 含义 |
|---|---|---|---|
| 1 | × 0.33 | **1**（`max(round(0.33),1)=1`） | n≤1 不缩放 |
| 3 | × 0.33 | **1**（`round(0.99)=1`） | 3 个 Bottleneck → 缩为 1 个 |
| 6 | × 0.33 | **2**（`round(1.98)=2`） | 6 个 Bottleneck → 缩为 2 个 |
| 9 | × 0.33 | **3**（`round(2.97)=3`） | 9 个 Bottleneck → 缩为 3 个 |

> 这就是为什么 yolov5s.yaml 里 C3 的 number 写着 3、6、9，但实际构建时只有 1、2、3 个 Bottleneck。

### Step C: 宽度系数调整

```python
if c2 != no:  # 如果不是检测头输出
    c2 = make_divisible(c2 * gw, 8)
```

`make_divisible(x, 8)` 将通道数调整为 8 的倍数（硬件对齐优化）。

| 配置 args[0] | gw=0.50 | 实际 c2 |
|---|---|---|
| 64 | × 0.50 = 32 | **32** |
| 128 | × 0.50 = 64 | **64** |
| 256 | × 0.50 = 128 | **128** |
| 512 | × 0.50 = 256 | **256** |
| 1024 | × 0.50 = 512 | **512** |

> 这就是 yaml 中写着千位级的通道数，实际模型却小得多的原因 ——**宽度系数 gw 把每个卷积层的通道都砍了一半**。

### Step D: 模块分类处理

```python
if m in {Conv, C3, SPPF, ...}:        # ① 标准卷积/C3 模块
    args = [c1, c2, *args[1:]]          # 用 c2 替换 args[0]
    if m in {C3, BottleneckCSP, ...}:
        args.insert(2, n)               # C3 额外插入 n（深度）
        n = 1                            # 因为 n 已经作为参数传进去了
elif m is nn.BatchNorm2d:              # ② BN 层
    args = [ch[f]]
elif m is Concat:                       # ③ 拼接层
    c2 = sum(ch[x] for x in f)          # 输出通道 = 各输入通道之和
elif m in {Detect, Segment}:           # ④ 检测头
    args.append([ch[x] for x in f])     # 传入各检测层的通道数
```

#### 以第 2 行 `[-1, 3, C3, [128]]` 为例

```
解析前:
  f=-1, n=3, m='C3', args=[128]

Step B: n = max(round(3 * 0.33), 1) = 1
Step C: c2 = make_divisible(128 * 0.50, 8) = 64

解析后:
  m = C3(c1=ch[-1]=64, c2=64, n=1)    ← 实际构建
  n = 1                                ← 因为 n 已经传给 C3 了
```

### Step E: 实例化模块

```python
m_ = nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)
```

- 如果 `n > 1`，用 `nn.Sequential` 包多个重复模块
- 如果 `n = 1`，直接实例化单个模块

### Step F: 附加元信息 + 更新通道列表

```python
m_.i, m_.f, m_.type, m_.np = i, f, t, np   # 记录索引、来源、类型、参数量
save.extend(...)                             # 记录需要保存中间特征的层
ch.append(c2)                                # 更新通道列表（供后续层使用）
```

> 元信息 `m_.f`（from）是实现跨层连接的关键——`BaseModel._forward_once()` 根据 `m.f` 决定从哪层取特征。

---

## 五、输入输出形状演算

以下展示 `yolov5s`（gd=0.33, gw=0.50）逐层的实际形状变化：

| 层 | 模块 | 输入 ch | 输出 ch | 空间尺寸 | 说明 |
|---|---|---|---|---|---|
| 0 | Conv | 3 | 32 | 320×320 | 起手卷积，stride=2 |
| 1 | Conv | 32 | 64 | 160×160 | stride=2 |
| 2 | **C3** | 64 | 64 | 160×160 | n=1, 特征提取 |
| 3 | Conv | 64 | 128 | 80×80 | stride=2 → **P3** |
| 4 | **C3** | 128 | 128 | 80×80 | n=2 |
| 5 | Conv | 128 | 256 | 40×40 | stride=2 → **P4** |
| 6 | **C3** | 256 | 256 | 40×40 | n=3 |
| 7 | Conv | 256 | 512 | 20×20 | stride=2 → **P5** |
| 8 | **C3** | 512 | 512 | 20×20 | n=1 |
| 9 | **SPPF** | 512 | 512 | 20×20 | 多尺度池化 |
| 10~23 | FPN+PAN 头 | ... | ... | ... | Neck 部分 |
| 24 | **Detect** | 128/256/512 | — | P3/P4/P5 | 三检测头输出 |

> 对比 yaml 中的通道数（64/128/256/512/1024）和实际通道数（32/64/128/256/512），就能直观感受到 `gw=0.50` 的效果。

---

## 六、与其他 YOLO 系列的对比

| 特性 | YOLOv3 (darknet) | YOLOv5 (PyTorch) |
|---|---|---|
| 模型定义方式 | 手写 `.cfg` 文件 + C 代码硬编码 | **Python `.yaml` + `parse_model()` 动态构建** |
| 缩放模型 | 重新设计整个网络 | 改 `gd`/`gw` 两个参数即可 |
| 添加新模块 | 写 C 代码 + 编译 | 在 `common.py` 写一个类，yaml 里引用 |
| 灵活性 | 低 | **极高** |

---

## 七、用人话总结

> **`parse_model()` 就像一家"自动化生产线"**：
> 1. 读一张 **"图纸"**（`.yaml`）—— 上面写着每层用什么零件、怎么连接
> 2. 按 **"缩放系数"**（`gd`, `gw`）自动调整零件的尺寸
> 3. 把每个零件实例化、打上标签（编号、来源、类型）
> 4. 最后把所有零件串成一条流水线（`nn.Sequential`）交付使用
>
> **想造大模型？** `gd=1.0, gw=1.0` → yolov5l
> **想造小模型？** `gd=0.33, gw=0.25` → yolov5n
> **想换 backbone？** 改 yaml 里的 `backbone` 列表即可
>
> **不改一行代码，只改一个 yaml 文件** —— 这就是 YOLOv5 工程化的精髓。


# 📉 YOLOv5 损失函数详解 — `utils/loss.py`

---

## 一、总览：三类损失加权求和

YOLOv5 的损失由三部分组成：

```
Loss = λ_box · L_box + λ_obj · L_obj + λ_cls · L_cls

λ_box = 0.05     （超参 hyp['box']）
λ_obj = 1.0      （超参 hyp['obj']）
λ_cls = 0.5      （超参 hyp['cls']）
```

| 损失 | 含义 | 计算方式 |
|---|---|---|
| **L_box** (定位损失) | 预测框和 GT 框的差距 | **CIoU Loss** |
| **L_obj** (置信度损失) | 该位置是否有目标 | **BCE With Logits** |
| **L_cls** (分类损失) | 目标类别是否正确 | **BCE With Logits** |

---

## 二、核心类 `ComputeLoss`

### 2.1 `__init__` — 准备工作

```python
class ComputeLoss:
    def __init__(self, model, autobalance=False):
        BCEcls = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([h['cls_pw']]))
        BCEobj = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([h['obj_pw']]))
```

初始化时主要做三件事：

1. **定义两个 BCE 损失函数** — 分别用于分类和置信度
2. **标签平滑** — `smooth_BCE()` 将硬标签 0/1 变为 `[0.1, 0.9]`，防止过拟合
3. **设置 obj 损失的平衡权重** — 针对 P3/P4/P5 三个检测层的不同权重：

```python
self.balance = {3: [4.0, 1.0, 0.4]}  # P3(小目标)权重最大, P5(大目标)权重最小
```

> 因为小目标（P3 层 80×80）的 grid 数量最多，每个 grid 包含目标的概率较低，所以用更大的权重补偿。

---

## 三、`__call__` — 前向传播中的损失计算

这是每次训练迭代都会执行的代码，分为 **5 个关键步骤**：

```mermaid
graph TD
    A["__call__(p, targets) 开始"] --> B["build_targets()<br/>正样本匹配"]
    B --> C["遍历 3 个检测层 P3/P4/P5"]
    C --> D["取出正样本对应的预测值"]
    D --> E["① L_box = CIoU Loss"]
    D --> F["② L_obj = BCE Loss<br/>(带 IoU 加权)"]
    D --> G["③ L_cls = BCE Loss<br/>(带标签平滑)"]
    E --> H["加权求和"]
    F --> H
    G --> H
    H --> I["返回 Loss"]
```

### Step 1: 取出正样本的预测值

```python
for i, pi in enumerate(p):  # p = [P3_out, P4_out, P5_out]
    b, a, gj, gi = indices[i]  # 正样本的: 图片索引, anchor索引, grid_y, grid_x
    pxy, pwh, _, pcls = pi[b, a, gj, gi].split((2, 2, 1, self.nc), 1)
```

`indices` 来自 `build_targets()`，标记了哪些位置是"正样本"。只有这些位置才参与 box_loss 和 cls_loss 的计算。

### Step 2: 定位损失 — CIoU (L_box)

```python
# 将预测值解码为实际坐标
pxy = pxy.sigmoid() * 2 - 0.5
pwh = (pwh.sigmoid() * 2) ** 2 * anchors[i]
pbox = torch.cat((pxy, pwh), 1)

# 计算 CIoU
iou = bbox_iou(pbox, tbox[i], CIoU=True)
lbox += (1.0 - iou).mean()
```

**CIoU 公式**（在 `bbox_iou()` 中实现）：

$$\text{CIoU} = \text{IoU} - \frac{\rho^2(b, b^{gt})}{c^2} - \alpha v$$

| 项 | 含义 | 惩罚什么 |
|---|---|---|
| $\text{IoU}$ | 交并比 | 重叠面积不够 |
| $\frac{\rho^2}{c^2}$ | 中心点距离 / 最小外接框对角线 | 中心点偏移 |
| $\alpha v$ | 长宽比一致性 | 形状与 GT 不一致 |

> **损失 = 1 - CIoU**，所以 CIoU 越大（框越准），损失越小。

### Step 3: 置信度损失 — BCE (L_obj)

```python
# 正样本的 target = CIoU 值 (越准 → 置信度target越高)
iou = iou.detach().clamp(0)
tobj[b, a, gj, gi] = iou  # target obj = IoU

# 负样本 target = 0
obji = self.BCEobj(pi[..., 4], tobj)  # 所有位置都参与
lobj += obji * self.balance[i]
```

**关键设计**：正样本的置信度 target 不是简单的 1，而是**预测框和 GT 的实际 IoU 值**。

> 这意味着：框得越准，置信度 target 越接近 1；框得一般，target 就低一些。
> 这样训练出来的模型，置信度能更真实地反映框的质量。

### Step 4: 分类损失 — BCE (L_cls)

```python
t = torch.full_like(pcls, self.cn)   # target 初始化为负类值
t[range(n), tcls[i]] = self.cp       # 正类位置设为平滑后的正类值
lcls += self.BCEcls(pcls, t)
```

- 使用 **标签平滑**（`smooth_BCE`）+ BCE 二分类损失
- 注意是**多标签二分类**（每个类别独立做 BCE），而不是 Softmax 多分类
- 这样做的好处：类别之间不互斥，允许一个目标同时属于多个类别

### Step 5: 加权求和

```python
lbox *= self.hyp['box']    # × 0.05
lobj *= self.hyp['obj']    # × 1.0
lcls *= self.hyp['cls']    # × 0.5
return (lbox + lobj + lcls) * bs  # × batch_size
```

---

## 四、`build_targets()` — 正样本匹配（最关键！）

这是 YOLOv5 相对 YOLOv3 改进最大的部分，决定了**哪些位置被当作正样本参与训练**。

### 4.1 流程概览

```mermaid
graph TD
    A["targets: (img, cls, x, y, w, h)"] --> B["Step 1: anchor 匹配<br/>wh 比例 < anchor_t"]
    B --> C["Step 2: 跨网格偏移<br/>中心点偏移 < 0.5 → 分配相邻 grid"]
    C --> D["Step 3: 生成 indices<br/>(b, a, gj, gi) 供损失计算"]
```

### 4.2 Step 1: Anchor 匹配

```python
r = t[..., 4:6] / anchors[:, None]   # GT宽高 / anchor宽高
j = torch.max(r, 1/r).max(2)[0] < self.hyp['anchor_t']  # anchor_t=4.0
```

- 计算每个 GT 的宽高与每个 anchor 的宽高比
- 如果比例在 `[1/4, 4]` 范围内，则认为匹配（`anchor_t=4.0`）
- 每个 GT 可能匹配 **1 个、2 个或 3 个** anchor

### 4.3 Step 2: 跨网格偏移

```python
# 判断 GT 中心点是否靠近 grid 边界
j, k = ((gxy % 1 < g) & (gxy > 1)).T  # 靠近左侧/上侧边界
l, m = ((gxi % 1 < g) & (gxi > 1)).T  # 靠近右侧/下侧边界

offsets = (torch.zeros_like(gxy)[None] + off[:, None])[j]
# off = [[0,0], [1,0], [0,1], [-1,0], [0,-1]]
```

```
GT 中心在蓝色 grid 中，但靠近右边界：
┌─────┬─────┬─────┐
│     │     │     │
├─────┼─────┼─────┤
│     │ ●→  │  ←  │  → 右侧 grid 也参与预测
├─────┼─────┼─────┤
│     │     │     │
└─────┴─────┴─────┘

最终每个 GT 匹配的正样本数：
  1 个 GT × 最多 3 个 anchor × 最多 3 个 grid = 最多 9 个正样本
```

### 4.4 对比 YOLOv3 的正样本匹配

| | YOLOv3 | YOLOv5 |
|---|---|---|
| Anchor 匹配 | Max IoU（1 个） | wh 比例 < anchor_t（最多 3 个） |
| Grid 分配 | 中心所在的 1 个 grid | 中心 ± 偏移（最多 3 个 grid） |
| 正样本数/GT | ~1~3 | ~3~9 |
| 小目标召回 | 低 | **显著提升** |

---

## 五、完整数据流示例

以一张 640×640 的图片、一个目标（person, x=200, y=150, w=80, h=120）为例：

```
① 目标归一化:
   targets = [0, 0, 200/640, 150/640, 80/640, 120/640]

② build_targets 匹配:
   P3/8  (80×80): grid=(25,18), anchor=(10×8=80, 13×8=104)  → 匹配 ✅
   P4/16 (40×40): grid=(12,9),  anchor=(30×16=480, 61×16=976) → 宽高比 >4, 不匹配
   P5/32 (20×20): grid=(6,4),   anchor=(116×32=3712, ...)      → 不匹配

③ 跨网格偏移:
   中心在 grid(25,18) 的 (0.3, 0.4) 处
   靠近上边界 → 上方 grid(25,17) 也参与
   总共: 2 anchor × 2 grid = 4 个正样本

④ 计算损失:
   L_box: 这 4 个位置预测框与 GT 的 CIoU
   L_obj: 这 4 个位置 target=CIoU, 其他 80×80×3-4=19196 个位置 target=0
   L_cls: 这 4 个位置 target=person
```

---

## 六、用人话总结

```mermaid
graph LR
    subgraph "ComputeLoss 内部"
        A["模型预测<br/>p = [P3, P4, P5]"] --> B["build_targets()<br/>找出正样本位置"]
        B --> C["L_box = CIoU<br/>框准不准？"]
        B --> D["L_obj = BCE<br/>这里有东西吗？"]
        B --> E["L_cls = BCE<br/>这是啥类别？"]
        C --> F["加权求和"]
        D --> F
        E --> F
    end
    F --> G["最终 Loss"]
```

> **L_box**：只看正样本，用 CIoU 衡量框得准不准
> **L_obj**：所有位置都看，正样本 target=CIoU（框得好才说好），负样本 target=0
> **L_cls**：只看正样本，用带平滑的 BCE 判断类别
> **build_targets**：每个 GT 匹配最多 3 个 anchor × 3 个相邻 grid = **最多 9 个正样本**


# 🚚 YOLOv5 数据加载详解 — `utils/dataloaders.py`

---

## 一、整体数据流

```mermaid
graph TD
    subgraph "磁盘"
        A["📁 数据集目录<br/>images/ + labels/"]
    end

    subgraph "初始化阶段 (一次)"
        B["扫描图片 + 标签<br/>验证格式"]
        C["缓存 labels → .cache<br/>下次秒加载"]
        D["可选: 缓存图片到 RAM<br/>大幅加速"]
    end

    subgraph "训练循环 (每轮)"
        E["__getitem__(index)"]
        F{"开启 mosaic?"}
        G["load_mosaic()<br/>4 图拼接"]
        H["load_image()<br/>单图 + letterbox"]
        I["增强: MixUp / HSV /<br/>翻转 / 透视 / Cutout"]
        J["转 Tensor + 归一化"]
    end

    A --> B
    B --> C
    C --> D
    D --> E
    E --> F
    F -->|是| G
    F -->|否| H
    G --> I
    H --> I
    I --> J
    J --> K["🚀 送入模型<br/>img: [3,640,640]<br/>labels: [N,6]"]
```

---

## 二、`create_dataloader()` — 入口函数

```python
def create_dataloader(path, imgsz, batch_size, stride, ...):
    dataset = LoadImagesAndLabels(path, imgsz, batch_size, augment=augment, ...)
    loader = InfiniteDataLoader if image_weights else DataLoader
    return loader(dataset, batch_size=batch_size, shuffle=shuffle, ...)
```

| 参数 | 作用 |
|---|---|
| `path` | 数据集路径（图片目录路径） |
| `imgsz` | 模型输入尺寸（默认 640） |
| `batch_size` | 每批图片数 |
| `augment` | 是否启用训练增强（训练=True, 验证=False） |
| `rect` | 矩形推理（按长宽比排序，减少 padding） |
| `cache` | 缓存模式：`False` / `'ram'` / `'disk'` |

> 💡 `InfiniteDataLoader` 是 YOLOv5 自定义的无尽迭代器，内部包装 `_RepeatSampler`，避免每个 epoch 重新创建 DataLoader worker，**减少 CPU 开销**。

---

## 三、`LoadImagesAndLabels` — 训练核心数据集

### 3.1 `__init__` — 初始化都做了什么？

**① 扫描图片文件**

```python
f += glob.glob(str(p / '**' / '*.*'), recursive=True)  # 递归找所有文件
self.im_files = sorted(x for x in f if x.split('.')[-1].lower() in IMG_FORMATS)  # 过滤图片
```

- 支持目录 / 文件列表 / glob 多种输入
- 只保留 `bmp, jpg, jpeg, png, tif, webp` 等格式

**② 标签验证 + 缓存**

```python
self.label_files = img2label_paths(self.im_files)  # images/xxx.jpg → labels/xxx.txt
# 尝试加载已有 .cache，否则重新扫描
cache, exists = np.load(cache_path, allow_pickle=True).item(), True  # 读缓存
if cache['hash'] == get_hash(self.label_files + self.im_files):  # 校验是否一致
    labels, shapes, segments = zip(*cache.values())  # 直接读缓存
else:
    cache = self.cache_labels(cache_path)  # 重新扫描
```

`img2label_paths()` 将图片路径转换为标签路径：

```python
# /datasets/coco128/images/train2017/000000000151.jpg
# → /datasets/coco128/labels/train2017/000000000151.txt
```

标签文件内容格式（每行一个目标）：
```
class_id  x_center  y_center  width  height
0         0.514     0.485     0.160  0.302
```

所有坐标均为 **归一化到 [0,1]** 的格式。

**③ 矩形训练 `rect`**

```python
if self.rect:
    ar = shapes[:, 1] / shapes[:, 0]  # 宽高比
    irect = ar.argsort()  # 按宽高比排序
    # 同一 batch 内图片宽高比相近 → 共享 letterbox padding
```

> 普通训练：所有图 resize 到 640×640（大量黑边浪费）
> 矩形训练：同一 batch 的图按宽高比分组 → **padding 大幅减少** → 训练加速 30%+

**④ 图片缓存到 RAM**

```python
if cache_images == 'ram':
    self.ims[i] = cv2.imread(self.im_files[i])  # 全部读到内存
```

> 缓存后训练时**零 IO 等待**，GPU 利用率拉满。但数据集太大时内存不够，会弹警告并自动禁用。

---

### 3.2 `__getitem__` — 每次迭代的核心

```python
def __getitem__(self, index):
    mosaic = self.mosaic and random.random() < hyp['mosaic']
    if mosaic:
        img, labels = self.load_mosaic(index)   # 🎯 4图拼接
        if random.random() < hyp['mixup']:
            img, labels = mixup(img, labels, *self.load_mosaic(...))  # 🎯 混合
    else:
        img, (h0, w0), (h, w) = self.load_image(index)  # 单图
        img, ratio, pad = letterbox(img, shape, ...)     # 等比例缩放 + padding

    # 坐标转换: xyxy → normalized xywh
    labels[:, 1:5] = xyxy2xywhn(labels[:, 1:5], ...)

    # 各种增强...
    augment_hsv(img, ...)       # HSV 色彩抖动
    random.flip(img, ...)       # 随机翻转
    self.albumentations(img)    # Albumentations 库增强

    return torch.from_numpy(img), labels_out, img_path, shapes
```

**输出格式：**

| 返回 | Shape | 说明 |
|---|---|---|
| `img` | `[3, 640, 640]` | RGB 图片 Tensor，归一化到 [0,1] |
| `labels_out` | `[N, 6]` | 每行: `[img_idx, class, x, y, w, h]` |
| `img_path` | `str` | 原始图片路径 |
| `shapes` | `tuple` | 原始尺寸 + 缩放信息（用于 mAP 计算） |

---

## 四、🎯 Mosaic 增强（最核心的增强）

### 4.1 什么是 Mosaic？

```
        原图 1 (左上)          原图 2 (右上)
         ┌──────────┐┌──────────┐
         │          ││          │
         │  ┌────┐  ││    ┌──┐  │
         │  │ 🐱 │  ││    │🚗│  │
         │  └────┘  ││    └──┘  │
    s    │          ││          │
    ←→   ├──────────┼┼──────────┤
         │          ││          │
         │     ┌──────────┐     │
         │     │ 🧑  │ 🐶│     │
         │     └──────────┘     │
         │          ││          │
         └──────────┘└──────────┘
        原图 3 (左下)          原图 4 (右下)

    ←─────── 2s = 1280 ───────→
```

- 将 4 张图随机拼成一张 **2×2 大图**（尺寸 1280×1280）
- 然后再随机裁剪 + resize 回 640×640

### 4.2 代码实现

```python
def load_mosaic(self, index):
    s = self.img_size  # 640
    yc, xc = random.uniform(-s//2, 3*s//2)  # 随机中心点
    indices = [index] + random.choices(self.indices, k=3)  # 当前图 + 3 张随机图

    for i, index in enumerate(indices):
        img, _, (h, w) = self.load_image(index)  # 加载单图

        if i == 0:  # 左上角
            img4 = np.full((s*2, s*2, 3), 114)  # 灰色底布
            # img4 上的位置
            x1a, y1a, x2a, y2a = max(xc-w,0), max(yc-h,0), xc, yc
            # img 上的裁剪区域
            x1b, y1b, x2b, y2b = w-(x2a-x1a), h-(y2a-y1a), w, h
        # ... 其他 3 个角类似

        img4[y1a:y2a, x1a:x2a] = img[y1b:y2b, x1b:x2b]  # 粘贴

    # 随机透视变换（裁剪 + 旋转 + 缩放 + 平移）
    img4 = random_perspective(img4, labels4, border=[-320, -320])
    return img4, labels4
```

### 4.3 Mosaic 的 4 大好处

| 好处 | 说明 |
|---|---|
| **变相增大 batch** | 每次看到 4 张图的内容 → 相当于 batch ×4，BN 更稳定 |
| **小目标更多** | 4 张图的小目标拼在一起 → 模型学习到更多小目标样本 |
| **上下文丰富** | 物体在"不应该出现"的背景中出现 → 增强泛化 |
| **平衡类别** | 随机混图 → 自然缓解类别不均衡 |

> 📊 消融实验：**去掉 Mosaic 后 mAP 下降约 3~5%**，是 YOLOv5 最重要的增强策略。

---

## 五、MixUp 增强

```python
if random.random() < hyp['mixup']:  # mixup=0.0 (默认关闭)
    img, labels = mixup(img, labels, *self.load_mosaic(...))
```

```mermaid
graph LR
    A["Mosaic 图 A"] --> C["λ * A + (1-λ) * B"]
    B["另一张 Mosaic 图 B"] --> C
    C --> D["标签也混合：<br/>A 的标签 + B 的标签"]
```

- 两张 mosaic 图按比例线性混合（λ 从 Beta 分布采样）
- 标签直接拼接（不是混合，因为目标检测中混合标签没意义）
- 默认关闭（`mixup=0.0`），开启时通常设 `0.1~0.5`

---

## 六、其他增强（`__getitem__` 中按序执行）

| 增强 | 超参 | 效果 |
|---|---|---|
| **Mosaic** | `mosaic=1.0` | 4 图拼接 ✅ |
| **MixUp** | `mixup=0.0` | 图混合（默认关） |
| **HSV 抖动** | `hsv_h/s/v` | 色调/饱和度/明度随机偏移 |
| **随机缩放** | `scale=0.5` | 0.5~1.5 倍随机缩放 |
| **随机平移** | `translate=0.1` | ±10% 平移 |
| **随机旋转** | `degrees=0.0` | 默认不旋转 |
| **随机翻转** | `fliplr=0.5` | 50% 水平翻转 |
| **Albumentations** | — | 额外的像素级增强 |

---

## 七、验证流程 vs 训练流程

| 阶段 | 增强 | Mosaic | Rect | Cache |
|---|---|---|---|---|
| **训练** | ✅ 全开 | ✅ | ❌（mosaic 冲突） | 可选 |
| **验证** | ❌ 仅 letterbox | ❌ | ✅（加速推理） | 可选 |

验证时 `__getitem__` 走的是非常简单路径：
```python
img, (h0, w0), (h, w) = self.load_image(index)         # 加载单图
img, ratio, pad = letterbox(img, self.img_size, ...)    # 等比例 + padding
labels = self.labels[index].copy()                       # 标签
labels[:, 1:] = xywhn2xyxy(...)                          # 转像素坐标
```

---

## 八、用人话总结

> **`dataloaders.py` = 图片处理流水线**
>
> 1. **初始化**：扫描文件夹 → 验证标签 → 缓存到 `.cache` → 下次秒开
> 2. **Mosaic**：4 图拼 1 张 → 变相翻倍 batch，小目标急救包
> 3. **增强流水线**：HSV → 翻转 → 平移 → 缩放 → Albumentations
> 4. **输出**：`[3,640,640]` Tensor + 归一化标签
> 5. **矩形训练**：同 batch 图按宽高比分组 → 省 padding → 训练提速
> 6. **InfiniteDataLoader**：不重启 worker → 省 CPU 开销


# 🏋️ YOLOv5 训练循环详解 — `train.py`

---

## 一、训练全流程总览

```mermaid
graph TD
    A["1. 初始化"] --> B["加载配置/模型/数据"]
    B --> C["2. 准备训练"]
    C --> D["优化器/Scheduler/EMA/AMP"]
    D --> E["3. 每轮训练"]
    E --> F["Warmup 预热"]
    F --> G["Multi-Scale<br/>多尺度训练"]
    G --> H["Forward 前向"]
    H --> I["Loss 损失计算"]
    I --> J["Backward 反向"]
    J --> K["梯度裁剪 + Optimizer.step"]
    K --> L["EMA 更新"]
    L --> M{"4. 每轮结束"}
    M --> N["验证 mAP"]
    M --> O["保存 last.pt / best.pt"]
    M --> P["EarlyStopping 检查"]
    N --> Q["5. 训练结束"]
    O --> Q
    P --> Q
    Q --> R["Strip Optimizer<br/>最终验证"]
```

---

## 二、阶段 1：初始化（`train()` 函数开头）

### 2.1 启动参数解析

```python
def parse_opt():
    parser = argparse.ArgumentParser()
    parser.add_argument('--weights', default='yolov5s.pt')   # 预训练权重
    parser.add_argument('--data', default='data/coco128.yaml') # 数据集配置
    parser.add_argument('--epochs', type=int, default=100)     # 训练轮数
    parser.add_argument('--batch-size', type=int, default=16)  # 批次大小
    parser.add_argument('--imgsz', type=int, default=640)      # 输入图片尺寸
    parser.add_argument('--device', default='')                # 设备
    parser.add_argument('--optimizer', default='SGD')          # 优化器
    parser.add_argument('--cos-lr', action='store_true')       # 余弦退火
    parser.add_argument('--patience', default=100)             # EarlyStopping
    parser.add_argument('--freeze', nargs='+', default=[0])    # 冻结层数
```

### 2.2 模型加载

```python
if pretrained:
    # 从预训练权重加载（迁移学习）
    ckpt = torch.load(weights, map_location='cpu')
    model = Model(cfg or ckpt['model'].yaml, ch=3, nc=nc)  # 创建新模型
    csd = intersect_dicts(ckpt['model'].float().state_dict(), model.state_dict(), exclude=['anchor'])
    model.load_state_dict(csd, strict=False)  # 只加载匹配的层
else:
    model = Model(cfg, ch=3, nc=nc)  # 从头训练
```

> 💡 `intersect_dicts` 取交集：预训练中与模型匹配的层才加载，不匹配的（如 nc 不同导致的分类层）自动忽略。

### 2.3 超参数缩放

```python
hyp['box'] *= 3 / nl           # 根据检测层数缩放 box 权重
hyp['cls'] *= nc / 80 * 3 / nl # 根据类别数缩放 cls 权重
hyp['obj'] *= (imgsz/640)**2 * 3 / nl  # 根据图像尺寸缩放 obj 权重
```

> 训练自定义数据集时（nc≠80 或 imgsz≠640），超参数自动适配，无需手动调参。

---

## 三、阶段 2：训练环境准备

| 组件 | 代码 | 作用 |
|---|---|---|
| **Dataloader** | `create_dataloader(..., augment=True)` | 训练数据加载 + Mosaic 等增强 |
| **验证 Loader** | `create_dataloader(..., rect=True)` | 验证数据（矩形推理） |
| **AutoAnchor** | `check_anchors(dataset, model)` | 自动优化锚框 |
| **Optimizer** | `smart_optimizer(model, 'SGD', lr0, momentum, wd)` | 智能优化器 |
| **Scheduler** | `LambdaLR` 余弦退火 or 线性衰减 | 学习率调度 |
| **EMA** | `ModelEMA(model)` | 指数移动平均（平滑参数） |
| **AMP** | `torch.cuda.amp.GradScaler` | 混合精度训练 |
| **ComputeLoss** | `ComputeLoss(model)` | 损失函数 |
| **EarlyStopping** | `EarlyStopping(patience=100)` | 早停 |

---

## 四、阶段 3：每轮训练（epoch 循环）

### 4.1 整轮结构

```python
for epoch in range(start_epoch, epochs):
    model.train()                              # ✅ 训练模式
    
    # --- 可选：图像加权采样 ---
    if opt.image_weights:
        iw = labels_to_image_weights(...)      # 难样本加权
    
    for i, (imgs, targets, paths, _) in enumerate(train_loader):
        # --- Warmup ---
        if ni <= nw:
            adjust_lr_and_momentum(...)        # 前 nw 步预热
        
        # --- Multi-Scale ---
        if opt.multi_scale:
            resize_to_random_size(...)          # 随机 320~960
        
        # --- Forward ---
        with torch.cuda.amp.autocast(amp):
            pred = model(imgs)                  # 前向传播
            loss, loss_items = compute_loss(pred, targets)  # 计算损失
        
        # --- Backward ---
        scaler.scale(loss).backward()           # 反向传播
        
        # --- Optimize (梯度累积) ---
        if ni - last_opt_step >= accumulate:
            scaler.unscale_(optimizer)
            clip_grad_norm_(model.parameters(), max_norm=10.0)  # 梯度裁剪
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if ema:
                ema.update(model)               # EMA 更新
    
    # --- 每轮结束 ---
    scheduler.step()                            # 学习率衰减
    if RANK in {-1, 0}:
        results = validate.run(...)              # ✅ 计算 mAP
        save_checkpoint(last, best)              # 保存权重
        if stopper(epoch, fi):
            break                                # 早停
```

### 4.2 Warmup 预热

```python
if ni <= nw:  # 前 nw 个 batch（约 3 轮）
    xi = [0, nw]
    # 学习率从 0 逐渐升到 lr0
    x['lr'] = np.interp(ni, xi, [hyp['warmup_bias_lr'] or 0.1, x['initial_lr'] * lf(epoch)])
    # 动量从 0.8 逐渐升到 0.937
    x['momentum'] = np.interp(ni, xi, [hyp['warmup_momentum'], hyp['momentum']])
    # 梯度累积步数从 1 逐渐升到 4（batch=16 时，nbs=64）
    accumulate = max(1, np.interp(ni, xi, [1, nbs / batch_size]).round())
```

> **为什么需要 warmup？** 刚启动时模型参数随机，直接大学习率可能导致梯度震荡。前 3 轮从 0 开始缓慢升温，让模型平稳起步。

### 4.3 Multi-Scale 多尺度训练

```python
if opt.multi_scale:
    sz = random.randrange(imgsz * 0.5, imgsz * 1.5 + gs) // gs * gs
    sf = sz / max(imgs.shape[2:])
    imgs = nn.functional.interpolate(imgs, size=ns, mode='bilinear', align_corners=False)
```

- 每批随机选择 `[320, 960]` 之间的一个尺寸
- 模型需要适应不同尺度的输入 → **尺度鲁棒性更强**
- 等价于额外的数据增强，约提升 1~2% mAP

### 4.4 梯度累积

```python
accumulate = max(round(nbs / batch_size), 1)  # nbs=64
# batch=16 → accumulate=4 （每 4 步更新一次）
# batch=64 → accumulate=1 （每步更新）
```

> 模拟"虚拟 batch_size = 64"，使小 batch 训练也能有稳定梯度。最终梯度 = 累积步数的梯度平均。

### 4.5 混合精度 AMP

```python
scaler = torch.cuda.amp.GradScaler(enabled=amp)   # 初始化
with torch.cuda.amp.autocast(amp):                  # 前向时自动半精度
    pred = model(imgs)
    loss, loss_items = compute_loss(pred, targets)
scaler.scale(loss).backward()                       # 反向前缩放损失防溢出
scaler.step(optimizer)                               # 更新参数
scaler.update()                                      # 更新缩放因子
```

> AMP 使大部分计算以 FP16 进行，**训练速度翻倍、显存减半**，精度损失通常 <0.5%。

---

## 五、阶段 4：验证与保存

### 5.1 验证

```python
results, maps, _ = validate.run(data_dict,
                                batch_size=batch_size // WORLD_SIZE * 2,
                                imgsz=imgsz,
                                half=amp,
                                model=ema.ema,       # 🎯 用 EMA 平滑模型验证
                                dataloader=val_loader,
                                compute_loss=compute_loss)
```

> 使用 EMA 模型（而非原始模型）验证，因为 EMA 参数是历史权重的指数平均，**更稳定、精度更高**。

### 5.2 Checkpoint 保存

```python
ckpt = {
    'epoch': epoch,                          # 当前轮数
    'best_fitness': best_fitness,            # 最佳分数
    'model': deepcopy(de_parallel(model)).half(),  # 原始模型
    'ema': deepcopy(ema.ema).half(),         # EMA 模型
    'updates': ema.updates,                  # EMA 更新次数
    'optimizer': optimizer.state_dict(),     # 优化器状态（可恢复训练）
    'opt': vars(opt),                        # 训练参数
    'date': datetime.now().isoformat()       # 时间戳
}
torch.save(ckpt, last)                       # 每轮覆盖 last.pt
if best_fitness == fi:
    torch.save(ckpt, best)                   # 最佳时存 best.pt
```

> `last.pt` 可恢复训练（含 optimizer 状态），`best.pt` 用于推理部署。

### 5.3 EarlyStopping

```python
stopper, stop = EarlyStopping(patience=opt.patience), False
# 每轮检查 fitness 是否提升
stop = stopper(epoch=epoch, fitness=fi)
if stop:
    break  # 连续 patience 轮未提升 → 提前停止
```

---

## 六、阶段 5：训练结束收尾

```python
# Strip optimizer 去掉优化器（减小文件体积）
strip_optimizer(f)  # 将 last.pt / best.pt 中的 optimizer 删除

# 最终验证 best.pt
validate.run(data_dict, model=attempt_load(best, device).half(), ...)
```

---

## 七、完整训练流程时间线

```mermaid
gantt
    title YOLOv5 训练时间线（epoch=100）
    dateFormat  X
    axisFormat  %d
    
    section 初始化
    加载数据+模型+缓存     : 0, 1
    
    section 训练
    Warmup (3 epochs)     : 1, 4
    正常训练 (Multi-Scale) : 4, 100
    
    section 每轮
    Train Epoch i          : 0, 1
    Validate               : 1, 1
    Save Checkpoint        : 1, 1
    
    section 结束
    最终验证+Strip         : 100, 101
```

---

## 八、关键参数速查

| 参数 | 默认值 | 含义 |
|---|---|---|
| `epochs` | 100 | 总训练轮数 |
| `batch-size` | 16 | 每批图片数（-1 自动测） |
| `imgsz` | 640 | 输入图片尺寸 |
| `optimizer` | SGD | Adam / AdamW / SGD |
| `cos-lr` | False | 余弦退火（默认线性衰减） |
| `freeze` | 0 | 冻结前 N 层 |
| `patience` | 100 | 早停耐心值 |
| `multi-scale` | False | 多尺度训练 |
| `label-smoothing` | 0.0 | 标签平滑 |
| `cache` | None | 缓存模式（ram/disk） |
| `image-weights` | False | 图像加权采样 |
| `quad` | False | 四合一 batch |

---

## 九、用人话总结

> 训练循环就像**健身**：
>
> 1. **Warmup（热身）** — 前 3 轮慢慢加运动量，防止拉伤
> 2. **Multi-Scale（变负荷训练）** — 有时举轻的有时举重的，肌肉适应性更强
> 3. **Gradient Accumulation（组间休息）** — 小重量多组数 = 小 batch 多累积
> 4. **AMP（半速跑）** — 用更少能量跑更快，体力消耗减半
> 5. **EMA（运动日记）** — 保留历史平均体重，比单次称重更准
> 6. **EarlyStopping（练到位了就停）** — 不再进步就收工，避免浪费时间
> 7. **每轮验一次 mAP（称体重）** — 看这轮练得怎么样，记录最佳状态
